# Modeling Readiness Assessor — Scanner

**Version**: 0.1.0  
**Purpose**: Enumerate Power BI semantic models and Fabric IQ ontologies in a Fabric workspace,
analyse modeling debt across four disciplines, and write a schema-versioned findings artifact
to OneLake.

**Prerequisites**:
- Fabric Viewer + Run permissions on the target workspace
- `scanner/lib/scanner/` attached as a custom library or mounted in the Fabric workspace
- `scoring-rubric.yaml` present in the notebook's working directory

**Run cells top-to-bottom.** Output from each cell is human-readable progress.

In [ ]:
# Cell 1: Imports and configuration
import sys, time, os, json
from pathlib import Path

# Make scanner library importable when running in Fabric notebook
# (adjust path if library is attached differently)
sys.path.insert(0, str(Path('.').resolve() / 'scanner' / 'lib'))

from scanner.findings import ScanScope
from scanner.artifact import FindingsArtifactWriter
from scanner.scoring import load_rubric, compute_score
from scanner.semantic_models import extract_semantic_models
from scanner.ontologies import extract_ontologies
from scanner.canonical_entity import detect_canonical_entity_conflicts
from scanner.field_lineage import audit_field_level_lineage
from scanner.layered_modeling import detect_layering_gaps, severity_for_gap as lm_severity, remediation_hint_for_gap as lm_hint
from scanner.steward_loop import detect_steward_loop_gaps, severity_for_gap as sl_severity, remediation_hint_for_gap as sl_hint

SCANNER_VERSION = '0.1.0'
RUBRIC_PATH = 'scoring-rubric.yaml'
ONELAKE_ROOT = 'Files/modeling-readiness'  # relative to workspace Files root

# ---- User configuration (edit these values before running) ----
WORKSPACE_ID = ''   # Fabric workspace GUID — required
WORKSPACE_URL = ''  # Full Fabric workspace URL — required
# ---------------------------------------------------------------

assert WORKSPACE_ID, 'Set WORKSPACE_ID to the target Fabric workspace GUID before running.'
assert WORKSPACE_URL, 'Set WORKSPACE_URL to the target Fabric workspace URL before running.'

print(f'Scanner v{SCANNER_VERSION} initialised.')
print(f'Target workspace: {WORKSPACE_ID}')
print(f'OneLake artifact root: {ONELAKE_ROOT}/')


In [ ]:
# Cell 2: Preview-feature dependency check (FR-013)
# Warns the user if Fabric IQ APIs are not available in this tenant.
import requests

FABRIC_IQ_AVAILABLE = True

try:
    token = notebookutils.credentials.getToken('fabric')  # type: ignore[name-defined]
    probe_url = f'https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/ontologies'
    probe_resp = requests.get(probe_url, headers={'Authorization': f'Bearer {token}'}, timeout=15)
    if probe_resp.status_code == 404:
        FABRIC_IQ_AVAILABLE = False
        print('⚠️  WARNING: Fabric IQ ontology API returned 404 for this workspace.')
        print('   Field-Level Lineage will be marked "not assessed in this version".')
        print('   Ensure Fabric IQ is enabled in your tenant settings if you need FLL scoring.')
    else:
        print('✅ Fabric IQ API reachable — Field-Level Lineage will be assessed.')
except Exception as e:
    FABRIC_IQ_AVAILABLE = False
    print(f'⚠️  WARNING: Could not probe Fabric IQ API: {e}')
    print('   Field-Level Lineage will be marked "not assessed in this version".')


In [ ]:
# Cell 3: Enumerate semantic models (FR-003)
t0 = time.perf_counter()

scope = ScanScope(type='full')
writer = FindingsArtifactWriter(
    output_root=ONELAKE_ROOT,
    workspace_id=WORKSPACE_ID,
    workspace_url=WORKSPACE_URL,
    scanner_version=SCANNER_VERSION,
    scope=scope,
)

print('Enumerating Power BI semantic models...')
pbi_token_fn = lambda: notebookutils.credentials.getToken('pbi')  # type: ignore[name-defined]
semantic_models = extract_semantic_models(WORKSPACE_ID, pbi_token_fn, raw_writer=writer)

elapsed = time.perf_counter() - t0
print(f'✅ Found {len(semantic_models)} semantic model(s) in {elapsed:.1f}s')
for m in semantic_models:
    print(f'   - {m.name} ({m.model_id}): {len(m.tables)} tables, {len(m.measures)} measures')
if elapsed > 60:
    print(f'⚠️  Semantic model enumeration exceeded 60s budget ({elapsed:.0f}s).')


In [ ]:
# Cell 4: Enumerate Fabric IQ ontologies (FR-004)
t0 = time.perf_counter()

if FABRIC_IQ_AVAILABLE:
    print('Enumerating Fabric IQ ontologies...')
    fabric_token_fn = lambda: notebookutils.credentials.getToken('fabric')  # type: ignore[name-defined]
    ontologies, ontology_status = extract_ontologies(WORKSPACE_ID, fabric_token_fn, raw_writer=writer)
    elapsed = time.perf_counter() - t0
    print(f'✅ Found {len(ontologies)} ontology(s) in {elapsed:.1f}s (status: {ontology_status})')
    for o in ontologies:
        print(f'   - {o.name} ({o.ontology_id}): {len(o.entity_types)} entity types')
    if elapsed > 60:
        print(f'⚠️  Ontology enumeration exceeded 60s budget ({elapsed:.0f}s).')
else:
    ontologies = []
    ontology_status = 'not_provisioned'
    print('ℹ️  Skipping ontology enumeration (Fabric IQ not available).')


In [ ]:
# Cell 5a: Canonical Entity Modeling analysis (FR-005/006) [wired from T023]
t0 = time.perf_counter()
print('Analysing Canonical Entity Modeling...')

cem_findings = []
cem_conflicts, cem_was_assessed = detect_canonical_entity_conflicts(
    models=semantic_models,
    ontologies=ontologies,
)
for conflict in cem_conflicts:
    from scanner.findings import Finding, SourceArtifactRef
    refs = [
        SourceArtifactRef(
            artifact_type=defn.source_type,
            artifact_id=defn.source_id,
            artifact_name=defn.source_name,
        )
        for defn in conflict.definitions
    ]
    cem_findings.append(
        Finding(
            finding_id=conflict.conflict_id,
            discipline='canonical_entity_modeling',
            severity='high',
            description=(
                f"'{conflict.logical_entity_name}' is defined inconsistently across "
                f"{len(conflict.definitions)} source artifact(s): "
                + ', '.join(d.source_name for d in conflict.definitions)
            ),
            source_artifacts=refs,
            remediation_hint='Align primary keys, join logic, and measure definitions across all source artifacts.',
            entity_name=conflict.logical_entity_name,
        )
    )

elapsed = time.perf_counter() - t0
print(f'✅ Canonical Entity Modeling: {len(cem_conflicts)} conflict(s) detected in {elapsed:.1f}s')
for c in cem_conflicts:
    print(f'   - {c.logical_entity_name}: {len(c.disagreements)} disagreement dimension(s)')
if elapsed > 60:
    print(f'⚠️  CEM analysis exceeded 60s budget ({elapsed:.0f}s).')


In [ ]:
# Cell 5b: Field-Level Lineage analysis (FR-007) [wired from T026]
t0 = time.perf_counter()
print('Auditing Field-Level Lineage source attribution...')

fll_findings = []
if ontologies:
    fll_gaps = audit_field_level_lineage(ontologies)
    for gap in fll_gaps:
        from scanner.findings import Finding, SourceArtifactRef
        fll_findings.append(
            Finding(
                finding_id=gap.gap_id,
                discipline='field_level_lineage',
                severity='medium',
                description=(
                    f"Entity type '{gap.entity_type_name}' is missing "
                    f"source-attribution properties: {', '.join(gap.missing_attribution_types)}"
                ),
                source_artifacts=[
                    SourceArtifactRef(
                        artifact_type='ontology',
                        artifact_id=gap.ontology_id,
                        artifact_name=next(
                            (o.name for o in ontologies if o.ontology_id == gap.ontology_id),
                            gap.ontology_id,
                        ),
                    )
                ],
                remediation_hint='Add source-system identifier, source-record identifier, extraction-timestamp, and confidence-score properties to this entity type.',
                entity_name=gap.entity_type_name,
            )
        )
    elapsed = time.perf_counter() - t0
    print(f'✅ Field-Level Lineage: {len(fll_gaps)} gap(s) detected in {elapsed:.1f}s')
    for g in fll_gaps:
        print(f'   - {g.entity_type_name}: missing {g.missing_attribution_types}')
    if elapsed > 60:
        print(f'⚠️  FLL analysis exceeded 60s budget ({elapsed:.0f}s).')
else:
    fll_gaps = []
    print('ℹ️  Skipping Field-Level Lineage analysis (no ontologies available).')


In [ ]:
# Cell 5c: Layered Modeling analysis (Discipline 3)
t0 = time.perf_counter()
print('Analysing Layered Modeling architecture...')

lm_findings = []
lm_gaps = detect_layering_gaps(semantic_models)
for gap in lm_gaps:
    from scanner.findings import Finding, SourceArtifactRef
    lm_findings.append(
        Finding(
            finding_id=gap.gap_id,
            discipline='layered_modeling',
            severity=lm_severity(gap),
            description=(
                f"Model '{gap.model_name}' ({gap.table_count} tables) has "
                + (f"no layer vocabulary detected" if not gap.detected_layers else
                   f"layer vocabulary for '{', '.join(gap.detected_layers)}' but missing '{', '.join(gap.missing_layers)}'")
            ),
            source_artifacts=[
                SourceArtifactRef(
                    artifact_type='semantic_model',
                    artifact_id=gap.model_id,
                    artifact_name=gap.model_name,
                )
            ],
            remediation_hint=lm_hint(gap),
            entity_name=gap.model_name,
        )
    )

elapsed = time.perf_counter() - t0
lm_has_signals = bool(semantic_models)  # always assessable when models exist
print(f'✅ Layered Modeling: {len(lm_gaps)} gap(s) in {elapsed:.1f}s')
for g in lm_gaps:
    print(f'   - {g.model_name}: detected={g.detected_layers}, missing={g.missing_layers}')
if elapsed > 60:
    print(f'⚠️  Layered Modeling analysis exceeded 60s budget ({elapsed:.0f}s).')


In [ ]:
# Cell 5d: Steward-Loop Modeling analysis (Discipline 4)
t0 = time.perf_counter()
print('Analysing Steward-Loop Modeling coverage...')

sl_findings = []
sl_gaps, sl_has_signals = detect_steward_loop_gaps(semantic_models, ontologies)
for gap in sl_gaps:
    from scanner.findings import Finding, SourceArtifactRef
    sl_findings.append(
        Finding(
            finding_id=gap.gap_id,
            discipline='steward_loop_modeling',
            severity=sl_severity(gap),
            description=(
                f"{gap.scope_type.replace('_', ' ').title()} '{gap.scope_name}' "
                f"is missing stewardship signals: {', '.join(gap.missing_signals)}"
                + (f" (found: {', '.join(gap.detected_signals)})" if gap.detected_signals else "")
            ),
            source_artifacts=[
                SourceArtifactRef(
                    artifact_type=gap.scope_type,
                    artifact_id=gap.scope_id,
                    artifact_name=gap.scope_name,
                )
            ],
            remediation_hint=sl_hint(gap),
            entity_name=gap.scope_name,
        )
    )

elapsed = time.perf_counter() - t0
if sl_has_signals:
    print(f'✅ Steward-Loop Modeling: {len(sl_gaps)} gap(s) in {elapsed:.1f}s')
    for g in sl_gaps:
        print(f'   - {g.scope_name} [{g.scope_type}]: missing {g.missing_signals}')
else:
    print('ℹ️  Steward-Loop Modeling: no stewardship vocabulary found — will be marked not_assessed.')
if elapsed > 60:
    print(f'⚠️  Steward-Loop analysis exceeded 60s budget ({elapsed:.0f}s).')


In [ ]:
# Cell 6: Compute maturity scores (FR-008)
rubric = load_rubric(RUBRIC_PATH)

all_findings = cem_findings + fll_findings + lm_findings + sl_findings

cem_score = compute_score('canonical_entity_modeling', len(cem_conflicts), rubric, has_signals=cem_was_assessed)
fll_score = compute_score(
    'field_level_lineage',
    len(fll_gaps) if ontologies else 0,
    rubric,
    has_signals=(ontology_status == 'assessed'),
)
lm_score = compute_score(
    'layered_modeling',
    len(lm_gaps),
    rubric,
    has_signals=lm_has_signals,
)
slm_score = compute_score(
    'steward_loop_modeling',
    len(sl_gaps),
    rubric,
    has_signals=sl_has_signals,
)

maturity_scores = [cem_score, fll_score, lm_score, slm_score]

print('\nMaturity Scores')
print('=' * 50)
for s in maturity_scores:
    score_display = str(s.score) if s.score is not None else 'N/A'
    print(f'{s.discipline:40s}  {score_display}/4  [{s.assessment_status}]')
    print(f'  {s.rationale}')


In [ ]:
# Cell 7: Write findings artifact to OneLake (FR-019)
t_total_start = time.perf_counter()

artifact_counts = {
    'semantic_models': len(semantic_models),
    'ontologies': len(ontologies),
}

run_dir = writer.write(
    findings=all_findings,
    maturity_scores=maturity_scores,
    artifact_counts=artifact_counts,
)

total_elapsed = time.perf_counter() - t_total_start
print(f'\n✅ Findings artifact written to: {run_dir}')
print(f'   run_id: {writer.run_id}')
print(f'   Findings: {len(all_findings)}')
print(f'   Maturity scores: {len(maturity_scores)}')
print(f'   Total write time: {total_elapsed:.1f}s')
print('\nNext step: open the narrator repository and run bootstrap.ps1 (or bootstrap.sh)')
print('then ask the GitHub Copilot agent about this workspace.')
